# Inferring the NK interdependency matrix from the truth table

N = 10, K = 1, 2, 4. No QCA: inference works on the real-valued truth table.

**Logic.** Row i adds Walsh weight only on subsets of its row set R_i = {i} ∪ N(i), and generically on all of them. With fixed K every R_i has size K + 1, so the maximal sets of the Walsh support are exactly the distinct row sets. What the table cannot say is which member of a set owns it. Any assignment of sets to members that uses every set gives the same support, and the i.i.d. contributions make all of them equally likely.

In [1]:
import itertools
import numpy as np
import pandas as pd
import warnings
from sklearn.linear_model import OrthogonalMatchingPursuit
warnings.filterwarnings("ignore")

N = 10
K_LIST = [1, 2, 4]
M = 100
rng = np.random.default_rng(1)

XINT = np.arange(2**N)
X = ((XINT[:, None] >> np.arange(N)[::-1]) & 1).astype(np.int8)       # column j = element j
BIT = 1 << np.arange(N)[::-1]                                          # subset mask bit of element j

def random_matrix(k, rng):
    A = np.eye(N, dtype=np.int8)
    for i in range(N):
        A[i, rng.choice([j for j in range(N) if j != i], k, replace=False)] = 1
    return A

def landscape(A, rng):
    f = np.zeros(2**N)
    for i in range(N):
        cols = np.flatnonzero(A[i])
        f += rng.random(2 ** len(cols))[X[:, cols] @ (2 ** np.arange(len(cols))[::-1])]
    return f / N

def walsh(f):
    h = f.copy(); s = 1
    while s < len(h):
        h = h.reshape(-1, 2, s)
        h = np.stack([h[:, 0] + h[:, 1], h[:, 0] - h[:, 1]], 1).reshape(-1); s *= 2
    return h / len(h)

def members(mask):
    return frozenset(j for j in range(N) if mask & BIT[j])

## 1. Full table: support, row sets, and every consistent matrix

`count_assignments` counts maps from elements to row sets that contain them and use every set (inclusion-exclusion). The edge posterior P(A_ij = 1) assumes a uniform prior over consistent matrices.

In [2]:
def maximal_sets(support_masks):
    """Maximal subsets of a down-closed family, as frozensets of elements."""
    S = [members(m) for m in support_masks]
    return [s for s in S if not any(s < t for t in S)]

def count_assignments(cands, F):
    """Number of maps i -> cands[i] (indices into F) whose image is all of F. Inclusion-exclusion over F."""
    tot = 0
    for r in range(len(F) + 1):
        for T in itertools.combinations(range(len(F)), r):
            T = set(T)
            tot += (-1) ** (len(F) - r) * np.prod([len(c & T) for c in cands], dtype=float)
    return tot

def infer_full(f, A):
    """Full truth table: Walsh support -> row-set family -> all matrices consistent with it."""
    w = walsh(f)
    F = maximal_sets(np.flatnonzero(np.abs(w) > 1e-12))
    K_hat = max(len(s) for s in F) - 1
    true_sets = {frozenset(np.flatnonzero(A[i])) for i in range(N)}
    cands = [{s for s, R in enumerate(F) if i in R} for i in range(N)]
    n = count_assignments(cands, F)
    # posterior P(A_ij = 1) under a uniform prior over consistent matrices
    P = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            c = list(cands); c[i] = {s for s in cands[i] if j in F[s]}
            P[i, j] = count_assignments(c, F) / n
    A_hat = (P > 0.5).astype(int)
    off = ~np.eye(N, dtype=bool)
    pairs_true = ((A.T.astype(int) @ A) > 0)[off]
    pairs_hat = np.array([[any(i in R and j in R for R in F) for j in range(N)] for i in range(N)])[off]
    return dict(K_hat=K_hat, sets_exact=set(F) == true_sets, pairs_exact=(pairs_hat == pairs_true).all(),
                n_consistent=n, unique=n == 1,
                edges_certain=((P == 0) | (P == 1))[off].mean(),
                edge_acc_map=(A_hat == A)[off].mean(),
                edge_acc_majority=max(K_hat / (N - 1), 1 - K_hat / (N - 1)))

## 2. Results, full table
- `sets_exact`, `pairs_exact`: row-set family and undirected interaction pairs recovered exactly.
- `unique`: only one matrix fits. `n_consistent_median`: how many fit.
- `edges_certain`: share of off-diagonal entries with posterior 0 or 1.
- `edge_acc_map`: accuracy of thresholding the posterior at 0.5, against always guessing the majority value.

In [3]:
rows = []
for k in K_LIST:
    for m in range(M):
        A = random_matrix(k, rng)
        rows.append(dict(K=k, **infer_full(landscape(A, rng), A)))
full = pd.DataFrame(rows)
full.groupby("K").agg(K_hat=("K_hat", "mean"),
                      sets_exact=("sets_exact", "mean"), pairs_exact=("pairs_exact", "mean"),
                      unique=("unique", "mean"), n_consistent_median=("n_consistent", "median"),
                      edges_certain=("edges_certain", "mean"), edge_acc_map=("edge_acc_map", "mean"),
                      edge_acc_majority=("edge_acc_majority", "mean")).round(3)

,K_hat,sets_exact,pairs_exact,unique,n_consistent_median,edges_certain,edge_acc_map,edge_acc_majority
K,,,,,,,,
1,1.0,1.0,1.0,0.0,2.0,0.892,0.958,0.889
2,2.0,1.0,1.0,0.0,39.5,0.572,0.856,0.778
4,4.0,1.0,1.0,0.0,3864.0,0.121,0.705,0.556


## 3. Partial table: sparse Walsh regression

Observe S of 1024 configurations. K is unknown: the basis holds every Walsh function up to order 5 (638 columns). Orthogonal matching pursuit fits until the residual vanishes. The maximal recovered sets give the row-set family and K.

In [4]:
PAR = np.array([bin(v).count("1") % 2 for v in range(2**N)])
K_MAX = 4
SUB = np.array([int(sum(BIT[list(c)])) for r in range(K_MAX + 2) for c in itertools.combinations(range(N), r)])
PHI = 1 - 2 * PAR[XINT[:, None] & SUB[None, :]]                      # Walsh basis up to order K_MAX + 1

def infer_partial(f, A, S, rng):
    idx = rng.choice(2**N, S, replace=False)
    c = OrthogonalMatchingPursuit(tol=1e-12 * S, fit_intercept=False).fit(PHI[idx], f[idx]).coef_
    F = maximal_sets(SUB[np.abs(c) > 1e-9])
    true = {frozenset(np.flatnonzero(A[i])) for i in range(N)}
    off = ~np.eye(N, dtype=bool)
    pt = ((A.T.astype(int) @ A) > 0)[off]
    ph = np.array([[any(i in R and j in R for R in F) for j in range(N)] for i in range(N)])[off]
    return dict(K_hat_ok=max(len(R) for R in F) - 1 == A[0].sum() - 1, sets_exact=set(F) == true,
                pair_precision=(ph & pt).sum() / max(ph.sum(), 1), pair_recall=(ph & pt).sum() / pt.sum())

rows = []
for k in K_LIST:
    for S in [64, 128, 256, 512]:
        for m in range(50):
            A = random_matrix(k, rng)
            rows.append(dict(K=k, S=S, **infer_partial(landscape(A, rng), A, S, rng)))
part = pd.DataFrame(rows)
part.groupby(["K", "S"]).mean().round(3)

K_hat_ok  sets_exact  pair_precision  pair_recall
K S                                                     
1 64       0.30        0.30           0.448        1.000
  128      1.00        1.00           1.000        1.000
  256      1.00        1.00           1.000        1.000
  512      1.00        1.00           1.000        1.000
2 64       0.00        0.00           0.522        0.999
  128      0.68        0.68           0.854        1.000
  256      1.00        1.00           1.000        1.000
  512      1.00        1.00           1.000        1.000
4 64       1.00        0.00           0.942        1.000
  128      1.00        0.00           0.928        1.000
  256      1.00        0.00           0.935        1.000
  512      1.00        1.00           1.000        1.000